# 04. 원문 탐색
- 각 Actor/Action별 실제 VOC 원문을 읽고 구체적인 Pain Point 발굴
- 기회영역 분석 결과에서 Underserved Area 중심으로 탐색

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
BASE_PATH = '/content/drive/Othercomputers/LG DX 노트북/LG_Dx_School/CX 프로젝트/가전 구독/가전 구독'
os.chdir(BASE_PATH)
!pwd

In [ ]:
import pandas as pd
import glob

# 모든 action pkl 로딩
df_list = []
for path in sorted(glob.glob('./cluster_df_*_action.pkl')):
    df_list.append(pd.read_pickle(path))

df = pd.concat(df_list, axis=0).reset_index(drop=True)
print(f'전체 데이터: {len(df)}건')
print(f'컬럼: {df.columns.tolist()}')

### 1. Action별 데이터 건수 확인

In [ ]:
# Actor/Action별 건수 확인
for actor in sorted(df['cluster'].unique()):
    actor_df = df[df['cluster'] == actor]
    print(f'\n[Actor {actor}]')
    for action in sorted(actor_df['action_cluster'].unique()):
        count = len(actor_df[actor_df['action_cluster'] == action])
        print(f'  Action {action}: {count}건')

### 2. 핵심 기회영역 원문 탐색
- **Actor0_Action1** (기회점수 19.73 — 압도적 1위)
- **Actor2_Action2** (기회점수 6.50 — 2위)

In [ ]:
def show_voc(df, actor, action, n=15):
    """
    특정 Actor/Action의 원문을 n개 출력
    플랫폼 정보와 함께 보여줌
    """
    filtered = df[(df['cluster'] == actor) & (df['action_cluster'] == action)].copy()
    print(f'\n==========================================')
    print(f'Actor{actor}_Action{action} | 전체 {len(filtered)}건 중 {n}개 샘플')
    print(f'==========================================')
    
    sample = filtered.sample(min(n, len(filtered)), random_state=42)
    
    for i, (_, row) in enumerate(sample.iterrows()):
        platform = row.get('플랫폼', '알수없음')
        keyword = row.get('키워드', '')
        text = str(row.get('full_text', row.get('review_clean', '')))
        
        print(f'\n[{i+1}번 | {platform} | 키워드: {keyword}]')
        print(text[:400])  # 400자까지만 출력
        print('...' if len(text) > 400 else '')
        print('-' * 50)

#### 🔴 Actor0_Action1 — 기회점수 1위 (중요도 10, 만족도 0.27)
**가전 구독/렌탈 비용·관리 고민 고객**

In [ ]:
show_voc(df, actor=0, action=1, n=15)

#### 🔴 Actor2_Action2 — 기회점수 2위 (중요도 3.25, 만족도 0.00)
**렌탈/구독 방법 비교·문의 고객**

In [ ]:
show_voc(df, actor=2, action=2, n=15)

### 3. 플랫폼별 필터링해서 보기
- 지식iN: 질문/답변 형태라 Pain Point가 가장 직접적으로 드러남
- 카페: 실제 사용 후기 중심

In [ ]:
def show_voc_by_platform(df, actor, action, platform='지식iN', n=10):
    """
    특정 플랫폼의 원문만 필터링해서 출력
    """
    filtered = df[
        (df['cluster'] == actor) & 
        (df['action_cluster'] == action) &
        (df['플랫폼'] == platform)
    ].copy()
    
    print(f'\n==========================================')
    print(f'Actor{actor}_Action{action} | {platform} | {len(filtered)}건 중 {n}개')
    print(f'==========================================')
    
    if len(filtered) == 0:
        print('해당 조건의 데이터가 없습니다.')
        return
    
    sample = filtered.sample(min(n, len(filtered)), random_state=42)
    
    for i, (_, row) in enumerate(sample.iterrows()):
        keyword = row.get('키워드', '')
        text = str(row.get('full_text', row.get('review_clean', '')))
        
        print(f'\n[{i+1}번 | 키워드: {keyword}]')
        print(text[:500])
        print('...' if len(text) > 500 else '')
        print('-' * 50)

# 지식iN에서 가장 날 것의 Pain Point 확인
show_voc_by_platform(df, actor=0, action=1, platform='지식iN', n=10)

In [ ]:
# 카페 후기도 확인
show_voc_by_platform(df, actor=0, action=1, platform='카페', n=10)

### 4. 키워드 포함 원문 검색
- 특정 단어가 포함된 원문만 골라서 보기
- Pain Point 키워드로 검색하면 더 날카로운 인사이트 발굴 가능

In [ ]:
def search_voc(df, actor, action, search_keywords, n=10):
    """
    특정 키워드가 포함된 원문만 검색
    search_keywords: 리스트로 입력 (OR 조건)
    """
    filtered = df[(df['cluster'] == actor) & (df['action_cluster'] == action)].copy()
    text_col = 'full_text' if 'full_text' in filtered.columns else 'review_clean'
    
    # 키워드 검색 (OR 조건)
    pattern = '|'.join(search_keywords)
    filtered = filtered[filtered[text_col].str.contains(pattern, na=False)]
    
    print(f'\n검색 키워드: {search_keywords}')
    print(f'Actor{actor}_Action{action} | 검색 결과: {len(filtered)}건')
    print('=' * 50)
    
    if len(filtered) == 0:
        print('검색 결과가 없습니다.')
        return
    
    sample = filtered.head(n)
    for i, (_, row) in enumerate(sample.iterrows()):
        text = str(row.get(text_col, ''))
        platform = row.get('플랫폼', '')
        print(f'\n[{i+1}번 | {platform}]')
        print(text[:500])
        print('...' if len(text) > 500 else '')
        print('-' * 50)

# Pain Point 키워드로 검색
search_voc(df, actor=0, action=1, 
           search_keywords=['위약금', '해지', '비싸', '부담', '모르', '복잡'], n=10)

In [ ]:
# 비용 관련 Pain Point
search_voc(df, actor=0, action=1,
           search_keywords=['계산', '비교', '일시불', '총비용', '얼마', '손해'], n=10)

In [ ]:
# 관리/케어 관련 Pain Point  
search_voc(df, actor=0, action=1,
           search_keywords=['방문', '케어', '필터', '교체', '언제', '안내'], n=10)

### 5. 인사이트 메모
- 위 원문들을 읽고 발견한 구체적인 Pain Point를 아래에 직접 정리하세요

```
예시)
- 고객 A: "6년 구독하면 총 얼마인지 계산이 안됨"
- 고객 B: "위약금이 얼마인지 사전에 안알려줌"
- 고객 C: "케어매니저가 언제 오는지 연락이 없음"
```